# HMM Viterbi From Scratch

In [ ]:
# ============================================================
# 06 - HMM VITERBI FROM SCRATCH
# ============================================================

import math

# States = POS tags
states = ["NN", "VB", "RB"]

# Start probabilities
start_prob = {
    "NN": 0.7,
    "VB": 0.2,
    "RB": 0.1
}

# Transition probabilities
transition_prob = {
    "NN": {"NN": 0.1, "VB": 0.7, "RB": 0.2},
    "VB": {"NN": 0.6, "VB": 0.1, "RB": 0.3},
    "RB": {"NN": 0.5, "VB": 0.2, "RB": 0.3}
}

# Emission probabilities
emission_prob = {
    "NN": {
        "dogs": 0.35,
        "cats": 0.35,
        "run": 0.03,
        "chase": 0.05,
        "quickly": 0.10,
        "slowly": 0.10
    },
    "VB": {
        "dogs": 0.03,
        "cats": 0.03,
        "run": 0.40,
        "chase": 0.40,
        "quickly": 0.10,
        "slowly": 0.02
    },
    "RB": {
        "dogs": 0.02,
        "cats": 0.02,
        "run": 0.02,
        "chase": 0.02,
        "quickly": 0.45,
        "slowly": 0.45
    }
}

# Small probability for an unknown word
UNKNOWN_PROB = 0.02

def emit(tag, word):
    return emission_prob[tag].get(word.lower(), UNKNOWN_PROB)

def viterbi(words):
    # dp[t][state] = best probability/log-probability
    # ending in 'state' at position t
    dp = []
    backpointer = []

    # --------------------------------------------------------
    # 1. INITIALIZATION
    # --------------------------------------------------------
    first = words[0]
    first_scores = {}
    first_back = {}

    for state in states:
        probability = start_prob[state] * emit(state, first)
        first_scores[state] = math.log(probability)
        first_back[state] = None

    dp.append(first_scores)
    backpointer.append(first_back)

    # --------------------------------------------------------
    # 2. RECURSION
    # --------------------------------------------------------
    for t in range(1, len(words)):
        word = words[t]
        scores = {}
        backs = {}

        for current_state in states:
            best_score = float("-inf")
            best_previous = None

            for previous_state in states:
                transition = transition_prob[previous_state][current_state]
                emission = emit(current_state, word)

                score = (
                    dp[t - 1][previous_state]
                    + math.log(transition)
                    + math.log(emission)
                )

                if score > best_score:
                    best_score = score
                    best_previous = previous_state

            scores[current_state] = best_score
            backs[current_state] = best_previous

        dp.append(scores)
        backpointer.append(backs)

    # --------------------------------------------------------
    # 3. TERMINATION
    # --------------------------------------------------------
    last_state = max(dp[-1], key=dp[-1].get)
    best_score = dp[-1][last_state]

    # --------------------------------------------------------
    # 4. BACKTRACKING
    # --------------------------------------------------------
    best_path = [last_state]

    for t in range(len(words) - 1, 0, -1):
        previous_state = backpointer[t][best_path[-1]]
        best_path.append(previous_state)

    best_path.reverse()

    return best_path, best_score

# Test
sentence = ["dogs", "run", "quickly"]

tags, log_probability = viterbi(sentence)

print("Sentence:")
print(sentence)

print("\nMost likely POS tag sequence:")
print(tags)

print("\nLog probability:")
print(log_probability)

print("\nWord / Tag:")
for word, tag in zip(sentence, tags):
    print(word, "/", tag)

# ============================================================
# VITERBI STEPS TO REMEMBER
# ============================================================
# 1. Initialization
# 2. Recursion
# 3. Termination
# 4. Backtracking
#
# Formula:
# V_t(s) = max(previous_state)
#          V_(t-1)(previous_state)
#          * transition(previous_state, s)
#          * emission(s, word_t)
#
# We use log probabilities in the code to avoid multiplying
# many very small numbers.

# ============================================================
# ============================================================
# states
# start_prob
# transition_prob
# emission_prob
# sentence
#


Sentence:
['dogs', 'run', 'quickly']

Most likely POS tag sequence:
['NN', 'VB', 'RB']

Log probability:
-5.8...

Word / Tag:
dogs / NN
run / VB
quickly / RB
